In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: IMPORTS & SETUP — GROQ API (Qwen3-32B)
# ═══════════════════════════════════════════════════════════════

import os
import json
import time
from typing import TypedDict, Annotated, List, Optional, Dict, Any
from datetime import datetime

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# ── Groq Client Setup ──
from groq import Groq

# Cấu hình API Key vào môi trường hệ thống để tăng tính bảo mật
GROQ_API_KEY = "YOUR_GROQ_API_KEY"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Khởi tạo client (Tự động nhận diện biến môi trường GROQ_API_KEY)
client = Groq()

# Wrapper function cải tiến để phối hợp linh hoạt với nhiều Agent Node khác nhau
def groq_chat(
    system_prompt: str, 
    user_prompt: str, 
    model: str = "qwen/qwen3-32b", 
    temperature: float = 0.6, 
    max_tokens: int = 4096
) -> str:
    """Gọi Groq API hỗ trợ truyền tham số linh hoạt cho từng Node cấu hình"""
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,            # Nhận giá trị động từ các node gọi đến
            max_completion_tokens=max_tokens,   # Khắc phục lỗi thiếu tham số đầu ra
            top_p=0.95,
            stream=False,
            stop=None
        )
        return completion.choices[0].message.content
    except Exception as e:
        print(f"❌ Groq API Error: {e}")
        print("⏳ Đang thử lại sau 2 giây...")
        time.sleep(2)
        
        # Vòng gỡ lỗi (Retry logic) phòng trường hợp nghẽn mạng hoặc quá tải dòng lệnh
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_completion_tokens=max_tokens,
            top_p=0.95,
            stream=False,
            stop=None
        )
        return completion.choices[0].message.content

print("✅ Cell 1: Groq API Setup hoàn tất")
print(f"   Model mặc định: qwen/qwen3-32b")
print(f"   API Key: {os.environ.get('GROQ_API_KEY')[:10]}...{os.environ.get('GROQ_API_KEY')[-4:]}")

✅ Cell 1: Groq API Setup hoàn tất
   Model mặc định: qwen/qwen3-32b
   API Key: YOUR_GROQ_API_KEY...B7EI


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: STATE DEFINITION (Robotic Marketer's Content Factory)
# ═══════════════════════════════════════════════════════════════

class ContentFactoryState(TypedDict):
    # Input
    client_brief: Dict[str, Any]
    
    # Node 1: Strategy
    content_brief: Optional[Dict[str, Any]]
    
    # Node 2: Research
    research_data: Optional[Dict[str, Any]]
    
    # Node 3: Creation (parallel outputs)
    long_form_draft: Optional[str]
    social_copies: Optional[Dict[str, Any]]
    visual_prompts: Optional[List[str]]
    
    # Node 4: SEO Optimization
    seo_optimized_content: Optional[Dict[str, Any]]
    
    # Node 5: Quality Assurance
    qa_score: Optional[float]
    qa_feedback: Optional[str]
    approved_content: Optional[Dict[str, Any]]
    
    # Node 6: Distribution
    distribution_plan: Optional[Dict[str, Any]]
    
    # Node 7: Analytics & Feedback
    performance_metrics: Optional[Dict[str, Any]]
    optimization_notes: Optional[str]
    
    # Control
    messages: Annotated[list, add_messages]
    current_step: str
    retry_count: int

# Test brief mẫu
TEST_BRIEF = {
    "brand_name": "TechStart Vietnam",
    "product": "Phần mềm quản lý dự án AI cho startup",
    "target_audience": "Founder startup Việt Nam, 25-40 tuổi, tech-savvy",
    "goal": "Tăng sign-up trial 200% trong Q3",
    "tone": "Chuyên nghiệp nhưng gần gũi, data-driven, có chút humor",
    "channels": ["blog", "linkedin", "email", "twitter"],
    "content_type": "long_form_blog",
    "word_count": 2500,
    "keywords": ["AI project management", "startup tools Vietnam", "quản lý dự án AI"],
    "competitors": ["Notion", "Asana", "Monday.com"],
    "brand_voice_examples": [
        "Chúng tôi không bán phần mềm. Chúng tôi bán thời gian.",
        "Startup không cần thêm công cụ. Startup cần ít công cụ hơn, hiệu quả hơn."
    ]
}

print("✅ Cell 2: State & Test Brief định nghĩa xong")
print(f"   Brand: {TEST_BRIEF['brand_name']}")
print(f"   Goal: {TEST_BRIEF['goal']}")

✅ Cell 2: State & Test Brief định nghĩa xong
   Brand: TechStart Vietnam
   Goal: Tăng sign-up trial 200% trong Q3


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: NODE 1 — STRATEGIST AGENT
# Tạo Content Brief chi tiết từ client input
# ═══════════════════════════════════════════════════════════════

STRATEGIST_SYSTEM_PROMPT = """Bạn là Chief Content Strategist đẳng cấp thế giới với 15 năm kinh nghiệm.
Nhiệm vụ: Tạo Content Brief chi tiết từ input khách hàng.

QUY TẮC:
1. Phân tích đối tượng sâu — không chỉ demographics mà còn psychographics
2. Xác định content angle khác biệt so với competitor
3. Keyword cluster phải bao gồm: 1 primary, 5 secondary, 10 LSI
4. Outline chi tiết theo Skeleton Method (Hook → Problem → Agitation → Solution → Proof → CTA)
5. Mỗi section có estimated word count và emotional trigger

OUTPUT JSON với cấu trúc:
{
  "title": "string",
  "topic": "string", 
  "target_audience": {"demographics": "...", "psychographics": "...", "pain_points": [...]},
  "tone_voice": "...",
  "content_angle": "Điểm khác biệt so với competitor...",
  "keyword_cluster": {"primary": "...", "secondary": [...], "lsi": [...]},
  "outline": [{"section": "...", "word_count": N, "trigger": "...", "key_points": [...]}],
  "visual_requirements": [{"type": "...", "description": "..."}],
  "distribution_strategy": {"channels": [...], "timing": "...", "repurpose_plan": "..."},
  "kpis": {"traffic_target": N, "conversion_target": N, "engagement_target": N}
}
"""

def strategist_node(state: ContentFactoryState):
    """Node 1: Tạo Content Brief từ Client Brief"""
    
    brief = state["client_brief"]
    
    user_prompt = f"""CLIENT BRIEF:
Brand: {brief['brand_name']}
Product: {brief['product']}
Target Audience: {brief['target_audience']}
Goal: {brief['goal']}
Tone: {brief['tone']}
Channels: {', '.join(brief['channels'])}
Content Type: {brief['content_type']}
Word Count: {brief['word_count']}
Keywords: {', '.join(brief['keywords'])}
Competitors: {', '.join(brief['competitors'])}
Brand Voice Examples: {brief['brand_voice_examples']}

Tạo Content Brief chi tiết. CHỈ trả về JSON, không thêm text khác."""
    
    response = groq_chat(STRATEGIST_SYSTEM_PROMPT, user_prompt, temperature=0.7, max_tokens=4096)
    
    try:
        content_brief = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        content_brief = {"raw_response": response, "parse_error": True}
    
    return {
        **state,
        "content_brief": content_brief,
        "current_step": "research"
    }

print("✅ Cell 3: Strategist Node định nghĩa xong")

✅ Cell 3: Strategist Node định nghĩa xong


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: NODE 2 — RESEARCHER AGENT  
# Thu thập intelligence: competitor, trends, data, quotes
# ═══════════════════════════════════════════════════════════════

RESEARCHER_SYSTEM_PROMPT = """Bạn là Senior Research Analyst chuyên phân tích thị trường và content.
Nhiệm vụ: Thu thập và tổng hợp dữ liệu chất lượng cao cho content creation.

QUY TẮC:
1. Phân tích 3-5 competitor content top ranking cho topic
2. Tìm statistics, data points, case studies có nguồn gốc rõ ràng
3. Xác định "semantic gap" — topics bị bỏ sót trong content hiện có
4. Thu thập quotes từ industry leaders
5. Tìm counter-intuitive insights — điều ngược lại common belief

OUTPUT JSON:
{
  "competitor_analysis": [
    {"competitor": "...", "content_title": "...", "strengths": [...], "weaknesses": [...], "gaps": [...]}
  ],
  "statistics": [{"claim": "...", "source": "...", "year": N}],
  "case_studies": [{"company": "...", "result": "...", "lesson": "..."}],
  "quotes": [{"author": "...", "quote": "...", "context": "..."}],
  "semantic_gaps": ["topic bị bỏ sót 1", "topic bị bỏ sót 2"],
  "trend_insights": ["trend 1", "trend 2"],
  "counter_intuitive_points": ["insight 1", "insight 2"],
  "recommended_sources": [...]
}
"""

def researcher_node(state: ContentFactoryState):
    """Node 2: Research & Intelligence Gathering"""
    
    brief = state["content_brief"]
    client = state["client_brief"]
    
    user_prompt = f"""CONTENT BRIEF:
Title: {brief.get('title', 'N/A')}
Topic: {brief.get('topic', 'N/A')}
Target Audience: {json.dumps(brief.get('target_audience', {}), ensure_ascii=False)}
Content Angle: {brief.get('content_angle', 'N/A')}
Outline: {json.dumps(brief.get('outline', [])[:5], ensure_ascii=False)}

COMPETITORS TO ANALYZE: {', '.join(client.get('competitors', []))}

Thực hiện research chi tiết. CHỈ trả về JSON."""
    
    response = groq_chat(RESEARCHER_SYSTEM_PROMPT, user_prompt, temperature=0.6, max_tokens=4096)
    
    try:
        research_data = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        research_data = {"raw_response": response, "parse_error": True}
    
    return {
        **state,
        "research_data": research_data,
        "current_step": "creation"
    }

print("✅ Cell 4: Researcher Node định nghĩa xong")

✅ Cell 4: Researcher Node định nghĩa xong


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: NODE 3 — CREATION AGENT
# Sản xuất: long-form + social copies + visual prompts
# ═══════════════════════════════════════════════════════════════

WRITER_SYSTEM_PROMPT = """Bạn là Senior Content Writer đoạt giải thưởng.
Viết content theo từng section, KHÔNG viết cả bài một lúc.

QUY TẮC VIẾT:
1. Mỗi section viết riêng biệt, focus vào 1 idea
2. Mở đầu bằng pattern: "Bạn có biết..." / "Hầu hết người ta nghĩ... nhưng sự thật là..." / "Năm 2023, một công ty đã..."
3. Thêm "texture": personal insight, counterintuitive claim, real example
4. Cite nguồn research khi dùng statistics
5. Transition mượt mà giữa các section
6. Kết thúc mỗi section bằng hook dẫn sang section tiếp theo
"""

COPYWRITER_SYSTEM_PROMPT = """Bạn là Conversion Copywriter chuyên social media.
Tạo platform-specific copies từ long-form content.

QUY TẮC:
- LinkedIn: Professional, data-forward, 150-200 words, 1 strong hook
- Twitter/X: Hook-first, <280 chars, thread-friendly
- Instagram: Conversational, 100 words, 5 hashtags, visual storytelling
- Email: Subject line + preview text + body (max 150 words)
- Ad: 3 variations với different angles (pain, gain, fear)

OUTPUT JSON với key cho từng platform.
"""

DESIGNER_SYSTEM_PROMPT = """Bạn là Visual Content Strategist.
Tạo image generation prompts từ content brief.

QUY TẮC:
- Mỗi prompt phải có: subject, style, mood, color palette, composition
- Tối ưu cho Midjourney v7 và FLUX.2
- Bao gồm: blog header, social graphics (1:1, 9:16, 16:9), infographic elements

OUTPUT: List of prompts với metadata.
"""

def creation_node(state: ContentFactoryState):
    """Node 3: Content Creation - Long-form + Social + Visual"""
    
    brief = state["content_brief"]
    research = state["research_data"]
    client = state["client_brief"]
    
    # 3A: Long-form Writer
    writer_prompt = f"""CONTENT BRIEF:
{json.dumps(brief, ensure_ascii=False)[:2000]}

RESEARCH DATA:
{json.dumps(research, ensure_ascii=False)[:3000]}

Viết bài hoàn chỉnh ~{client.get('word_count', 2000)} từ. 
Structure theo outline. Thêm citations từ research."""
    
    long_form = groq_chat(WRITER_SYSTEM_PROMPT, writer_prompt, temperature=0.7, max_tokens=4096)
    
    # 3B: Copywriter
    copy_prompt = f"""LONG-FORM CONTENT (tóm tắt):
{long_form[:2000]}...

Tạo copies cho channels: {', '.join(client.get('channels', []))}
CHỈ trả về JSON."""
    
    copy_response = groq_chat(COPYWRITER_SYSTEM_PROMPT, copy_prompt, temperature=0.7, max_tokens=4096)
    
    try:
        social_copies = json.loads(copy_response.replace("```json", "").replace("```", "").strip())
    except:
        social_copies = {"raw": copy_response}
    
    # 3C: Visual Prompts
    design_prompt = f"""CONTENT: {brief.get('title', '')}
VISUAL REQUIREMENTS: {json.dumps(brief.get('visual_requirements', []), ensure_ascii=False)}

Tạo 5 image prompts. CHỈ trả về JSON list."""
    
    design_response = groq_chat(DESIGNER_SYSTEM_PROMPT, design_prompt, temperature=0.7, max_tokens=2048)
    
    try:
        visual_prompts = json.loads(design_response.replace("```json", "").replace("```", "").strip())
    except:
        visual_prompts = [design_response[:500]]
    
    return {
        **state,
        "long_form_draft": long_form,
        "social_copies": social_copies,
        "visual_prompts": visual_prompts,
        "current_step": "seo"
    }

print("✅ Cell 5: Creation Node định nghĩa xong")

✅ Cell 5: Creation Node định nghĩa xong


In [6]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: NODE 4 — SEO OPTIMIZER
# Optimize content cho search engines
# ═══════════════════════════════════════════════════════════════

SEO_SYSTEM_PROMPT = """Bạn là Technical SEO Expert với 10 năm kinh nghiệm.
Optimize content cho organic search performance.

QUY TẮC:
1. Title tag: 50-60 chars, primary keyword ở đầu, power word, number nếu có thể
2. Meta description: 150-160 chars, CTA, primary + secondary keywords
3. Header structure: H1 (1), H2 (3-5), H3 (5-10), logical hierarchy
4. Keyword placement: Primary trong first 100 words, H1, H2 đầu tiên
5. Internal linking: 3-5 suggestions với anchor text tự nhiên
6. Schema markup: Recommend structured data type
7. Readability: Flesch score 60-70, sentence length <20 words, paragraph <4 lines
8. Image alt text: Descriptive, include keyword nếu tự nhiên

OUTPUT JSON:
{
  "title_tag": "...",
  "meta_description": "...",
  "header_structure": [{"level": "H1", "text": "...", "keyword_included": true}],
  "keyword_placement": {"primary_positions": [...], "secondary_positions": [...], "density": "X%"},
  "internal_links": [{"anchor": "...", "target": "...", "context": "..."}],
  "schema_markup": {"type": "...", "properties": {...}},
  "readability_score": {"flesch": N, "grade_level": "..."},
  "optimized_content": "full content with headers and formatting",
  "seo_recommendations": [...]
}
"""

def seo_optimizer_node(state: ContentFactoryState):
    """Node 4: SEO Optimization"""
    
    draft = state["long_form_draft"]
    brief = state["content_brief"]
    
    user_prompt = f"""CONTENT DRAFT:
{draft[:4000]}...

TARGET KEYWORDS:
Primary: {brief.get('keyword_cluster', {}).get('primary', 'N/A')}
Secondary: {brief.get('keyword_cluster', {}).get('secondary', [])}

Optimize content. CHỈ trả về JSON."""
    
    response = groq_chat(SEO_SYSTEM_PROMPT, user_prompt, temperature=0.5, max_tokens=4096)
    
    try:
        seo_data = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        seo_data = {"raw_response": response, "parse_error": True}
    
    return {
        **state,
        "seo_optimized_content": seo_data,
        "current_step": "qa"
    }

print("✅ Cell 6: SEO Optimizer Node định nghĩa xong")

✅ Cell 6: SEO Optimizer Node định nghĩa xong


In [7]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: NODE 5 — QUALITY ASSURANCE
# Kiểm tra chất lượng, brand consistency, fact-check
# ═══════════════════════════════════════════════════════════════

EDITOR_SYSTEM_PROMPT = """Bạn là Chief Content Editor khắt khe nhất ngành.
Đánh giá content theo 10 tiêu chí, cho điểm từng tiêu chí 0-10.

TIÊU CHÍ ĐÁNH GIÁ:
1. Brand Voice Consistency: Tone có khớp brand voice examples không?
2. Factual Accuracy: Statistics, claims có verify được không?
3. SEO Optimization: Keywords tự nhiên, structure tốt?
4. Engagement Potential: Có hook, storytelling, emotional trigger?
5. Clarity & Structure: Dễ đọc, logical flow, scannable?
6. Originality: Khác biệt so với competitor, không generic?
7. Call-to-Action: CTA rõ ràng, compelling, relevant?
8. Visual-Text Alignment: Visual prompts khớp content?
9. Platform Suitability: Social copies phù hợp từng platform?
10. Grammar & Style: Không lỗi, style consistent?

QUY TẮC:
- Score < 5: REJECT, return to Creation with detailed feedback
- Score 5-7: CONDITIONAL PASS, minor revisions needed
- Score 8-10: APPROVED

OUTPUT JSON:
{
  "scores": {"brand_voice": N, "factual": N, "seo": N, "engagement": N, "clarity": N, "originality": N, "cta": N, "visual": N, "platform": N, "grammar": N},
  "overall_score": N.N,
  "verdict": "APPROVED|CONDITIONAL|REJECT",
  "feedback": {"strengths": [...], "weaknesses": [...], "action_items": [...]},
  "revision_needed": true|false
}
"""

def qa_node(state: ContentFactoryState):
    """Node 5: Quality Assurance"""
    
    seo_content = state["seo_optimized_content"]
    social = state["social_copies"]
    visual = state["visual_prompts"]
    client = state["client_brief"]
    
    user_prompt = f"""CONTENT TO EVALUATE:
SEO Content: {json.dumps(seo_content, ensure_ascii=False)[:2000]}...
Social Copies: {json.dumps(social, ensure_ascii=False)[:1500]}...
Visual Prompts: {json.dumps(visual, ensure_ascii=False)[:1000]}...

BRAND VOICE EXAMPLES: {client.get('brand_voice_examples', [])}

Đánh giá chi tiết. CHỈ trả về JSON."""
    
    response = groq_chat(EDITOR_SYSTEM_PROMPT, user_prompt, temperature=0.3, max_tokens=4096)
    
    try:
        qa_result = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        qa_result = {"overall_score": 0, "verdict": "REJECT", "raw": response}
    
    overall = qa_result.get("overall_score", 0)
    verdict = qa_result.get("verdict", "REJECT")
    
    if overall >= 8.0 and verdict == "APPROVED":
        return {
            **state,
            "qa_score": overall,
            "qa_feedback": qa_result.get("feedback", {}),
            "approved_content": {
                "seo": seo_content,
                "social": social,
                "visual": visual
            },
            "current_step": "distribute"
        }
    else:
        return {
            **state,
            "qa_score": overall,
            "qa_feedback": qa_result.get("feedback", {}),
            "current_step": "creation",
            "retry_count": state.get("retry_count", 0) + 1
        }

print("✅ Cell 7: QA Node định nghĩa xong")

✅ Cell 7: QA Node định nghĩa xong


In [13]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: NODE 6 — DISTRIBUTION
# Tạo distribution plan + platform-specific formatting
# ═══════════════════════════════════════════════════════════════

DISTRIBUTOR_SYSTEM_PROMPT = """Bạn là Distribution Strategist chuyên multi-channel publishing.
Tạo distribution plan chi tiết với timeline và platform optimization.

QUY TẮC:
1. Staggered release: Blog first → Social day+1 → Email day+3 → Repurpose day+7
2. Mỗi platform có custom formatting: hashtags, mentions, link placement
3. A/B test plan cho headlines và visuals
4. Engagement strategy: best posting times, reply templates, community management
5. Tracking: UTM parameters, conversion goals, attribution

OUTPUT JSON:
{
  "distribution_timeline": [
    {"day": 0, "channel": "blog", "action": "publish", "time": "...", "content": "..."},
    {"day": 1, "channel": "linkedin", "action": "post", "time": "...", "content": "..."}
  ],
  "platform_formatting": {
    "linkedin": {"hashtags": [...], "mentions": [...], "format": "..."},
    "twitter": {"thread_plan": [...], "hashtags": [...]},
    "email": {"subject_lines": [...], "preview_text": "...", "send_time": "..."}
  },
  "ab_test_plan": {"headlines": [...], "visuals": [...], "metrics": [...]},
  "engagement_strategy": {"reply_templates": [...], "community_hooks": [...]},
  "tracking_setup": {"utm_params": {...}, "conversion_goals": [...], "attribution_model": "..."}
}
"""

def distributor_node(state: ContentFactoryState):
    """Node 6: Distribution Planning"""
    
    approved = state["approved_content"]
    client = state["client_brief"]
    
    user_prompt = f"""APPROVED CONTENT:
SEO: {json.dumps(approved.get('seo', {}), ensure_ascii=False)[:1500]}...
Social: {json.dumps(approved.get('social', {}), ensure_ascii=False)[:1000]}...

TARGET CHANNELS: {', '.join(client.get('channels', []))}

Tạo distribution plan. CHỈ trả về JSON."""
    
    response = groq_chat(DISTRIBUTOR_SYSTEM_PROMPT, user_prompt, temperature=0.6, max_tokens=4096)
    
    try:
        dist_plan = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        dist_plan = {"raw": response}
    
    return {
        **state,
        "distribution_plan": dist_plan,
        "current_step": "analytics"
    }

print("✅ Cell 8: Distribution Node định nghĩa xong")

✅ Cell 8: Distribution Node định nghĩa xong


In [14]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: NODE 7 — ANALYTICS & FEEDBACK
# Performance tracking + optimization recommendations + feedback loop
# ═══════════════════════════════════════════════════════════════

ANALYST_SYSTEM_PROMPT = """Bạn là Marketing Analytics Expert.
Tạo performance framework và optimization recommendations.

QUY TẮC:
1. Define KPIs cho từng channel và content type
2. Benchmark against industry standards
3. Identify optimization opportunities
4. Content refresh strategy: khi nào update, cái gì cần thay đổi
5. Feedback loop: insights từ data để cải thiện workflow tiếp theo

OUTPUT JSON:
{
  "kpis": {
    "traffic": {"target": N, "metrics": ["organic", "direct", "social", "referral"]},
    "engagement": {"target": N, "metrics": ["time_on_page", "bounce_rate", "scroll_depth", "shares"]},
    "conversions": {"target": N, "metrics": ["cta_clicks", "signups", "purchases"]},
    "seo": {"target": N, "metrics": ["rankings", "backlinks", "ctr"]}
  },
  "benchmarks": {"industry_avg": {...}, "top_performer": {...}},
  "optimization_opportunities": [
    {"area": "...", "issue": "...", "recommendation": "...", "expected_impact": "...", "priority": "high|medium|low"}
  ],
  "content_refresh_plan": {"timeline": "...", "updates": [...]},
  "feedback_for_next_cycle": {
    "what_worked": [...],
    "what_didnt": [...],
    "prompt_improvements": [...],
    "workflow_adjustments": [...]
  }
}
"""

def analytics_node(state: ContentFactoryState):
    """Node 7: Analytics & Continuous Optimization"""
    
    dist_plan = state["distribution_plan"]
    brief = state["content_brief"]
    
    user_prompt = f"""DISTRIBUTION PLAN: {json.dumps(dist_plan, ensure_ascii=False)[:1500]}...
CONTENT GOALS: {brief.get('kpis', {})}

Tạo analytics framework. CHỈ trả về JSON."""
    
    response = groq_chat(ANALYST_SYSTEM_PROMPT, user_prompt, temperature=0.5, max_tokens=4096)
    
    try:
        analytics = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        analytics = {"raw": response}
    
    return {
        **state,
        "performance_metrics": analytics.get("kpis", {}),
        "optimization_notes": analytics.get("feedback_for_next_cycle", {}),
        "current_step": "complete"
    }

print("✅ Cell 9: Analytics Node định nghĩa xong")

✅ Cell 9: Analytics Node định nghĩa xong


In [10]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: ORCHESTRATOR
# Chạy 7 nodes theo thứ tự với conditional retry logic
# ═══════════════════════════════════════════════════════════════

def run_content_factory(client_brief: dict) -> ContentFactoryState:
    """Chạy full pipeline: Strategist → Researcher → Creation → SEO → QA → Distributor → Analytics"""
    
    # Initial state
    state = {
        "client_brief": client_brief,
        "content_brief": None,
        "research_data": None,
        "long_form_draft": None,
        "social_copies": None,
        "visual_prompts": None,
        "seo_optimized_content": None,
        "qa_score": None,
        "qa_feedback": None,
        "approved_content": None,
        "distribution_plan": None,
        "performance_metrics": None,
        "optimization_notes": None,
        "current_step": "strategist",
        "retry_count": 0
    }
    
    print("🚀 BẮT ĐẦU CONTENT FACTORY PRO")
    print("=" * 60)
    
    # Node 1: Strategist
    print("\n📌 [1/7] STRATEGIST — Tạo Content Brief...")
    state = strategist_node(state)
    print(f"   ✅ Title: {state['content_brief'].get('title', 'N/A')[:60]}...")
    
    # Node 2: Researcher
    print("\n🔍 [2/7] RESEARCHER — Thu thập intelligence...")
    state = researcher_node(state)
    research = state['research_data']
    print(f"   ✅ {len(research.get('competitor_analysis', []))} competitors, {len(research.get('statistics', []))} stats, {len(research.get('semantic_gaps', []))} gaps")
    
    # Node 3: Creation
    print("\n✍️ [3/7] CREATION — Viết content...")
    state = creation_node(state)
    print(f"   ✅ Draft: {len(state['long_form_draft'] or '')} chars")
    print(f"   ✅ Social copies: {list(state['social_copies'].keys())[:5]}")
    print(f"   ✅ Visual prompts: {len(state['visual_prompts'])}")
    
    # Node 4: SEO
    print("\n🔍 [4/7] SEO OPTIMIZER — Optimize cho search...")
    state = seo_optimizer_node(state)
    seo = state['seo_optimized_content']
    print(f"   ✅ Title tag: {seo.get('title_tag', 'N/A')[:50]}...")
    print(f"   ✅ Flesch: {seo.get('readability_score', {}).get('flesch', 'N/A')}")
    
    # Node 5: QA (với retry loop)
    print("\n✅ [5/7] QUALITY ASSURANCE — Kiểm tra chất lượng...")
    max_retries = 2
    for attempt in range(max_retries + 1):
        state = qa_node(state)
        score = state['qa_score']
        print(f"   Attempt {attempt + 1}: Score {score}/10 — {state.get('current_step', '').upper()}")
        
        if state['current_step'] == 'distribute':
            break
        elif state['retry_count'] >= max_retries:
            print("   ⚠️ Max retries reached. Proceeding with current version.")
            state['current_step'] = 'distribute'
            break
    
    # Node 6: Distribution
    print("\n📤 [6/7] DISTRIBUTION — Tạo publish plan...")
    state = distributor_node(state)
    dist = state['distribution_plan']
    print(f"   ✅ {len(dist.get('distribution_timeline', []))} scheduled posts")
    
    # Node 7: Analytics
    print("\n📊 [7/7] ANALYTICS — Performance framework...")
    state = analytics_node(state)
    opt = state['optimization_notes']
    print(f"   ✅ {len(opt.get('what_worked', []))} insights, {len(state.get('performance_metrics', {}))} KPI categories")
    
    print("\n" + "=" * 60)
    print("✅ WORKFLOW HOÀN TẤT")
    print("=" * 60)
    
    return state

print("✅ Cell 10: Orchestrator định nghĩa xong")
print("   Chạy: result = run_content_factory(TEST_BRIEF)")

✅ Cell 10: Orchestrator định nghĩa xong
   Chạy: result = run_content_factory(TEST_BRIEF)


In [15]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: RUN FULL WORKFLOW
# Chạy với test brief
# ═══════════════════════════════════════════════════════════════

# Initial state
initial_state = {
    "client_brief": TEST_BRIEF,
    "content_brief": None,
    "research_data": None,
    "long_form_draft": None,
    "social_copies": None,
    "visual_prompts": None,
    "seo_optimized_content": None,
    "qa_score": None,
    "qa_feedback": None,
    "approved_content": None,
    "distribution_plan": None,
    "performance_metrics": None,
    "optimization_notes": None,
    "messages": [],
    "current_step": "strategist",
    "retry_count": 0
}
# ═══════════════════════════════════════════════════════════════
# CELL 11: RUN FULL WORKFLOW
# ═══════════════════════════════════════════════════════════════

result = run_content_factory(TEST_BRIEF)

🚀 BẮT ĐẦU CONTENT FACTORY PRO

📌 [1/7] STRATEGIST — Tạo Content Brief...
   ✅ Title: N/A...

🔍 [2/7] RESEARCHER — Thu thập intelligence...
   ✅ 0 competitors, 0 stats, 0 gaps

✍️ [3/7] CREATION — Viết content...
   ✅ Draft: 10351 chars
   ✅ Social copies: ['raw']
   ✅ Visual prompts: 1

🔍 [4/7] SEO OPTIMIZER — Optimize cho search...
   ✅ Title tag: N/A...
   ✅ Flesch: N/A

✅ [5/7] QUALITY ASSURANCE — Kiểm tra chất lượng...
   Attempt 1: Score 0/10 — CREATION
   Attempt 2: Score 0/10 — CREATION
   ⚠️ Max retries reached. Proceeding with current version.

📤 [6/7] DISTRIBUTION — Tạo publish plan...


AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12: INSPECT OUTPUT DETAIL
# ═══════════════════════════════════════════════════════════════

print("🔍 CHI TIẾT OUTPUT\n")

# 1. Content Brief
print("━" * 50)
print("📌 NODE 1: CONTENT BRIEF")
print("━" * 50)
brief = result.get("content_brief", {})
print(f"Title: {brief.get('title', 'N/A')}")
print(f"Topic: {brief.get('topic', 'N/A')}")
print(f"Content Angle: {brief.get('content_angle', 'N/A')[:100]}...")
print(f"Primary Keyword: {brief.get('keyword_cluster', {}).get('primary', 'N/A')}")
print(f"Outline sections: {len(brief.get('outline', []))}")

# 2. Research
print("\n" + "━" * 50)
print("🔍 NODE 2: RESEARCH DATA")
print("━" * 50)
research = result.get("research_data", {})
print(f"Competitors analyzed: {len(research.get('competitor_analysis', []))}")
print(f"Statistics: {len(research.get('statistics', []))}")
print(f"Case studies: {len(research.get('case_studies', []))}")
print(f"Semantic gaps: {research.get('semantic_gaps', [])[:3]}")

# 3. Creation
print("\n" + "━" * 50)
print("✍️ NODE 3: CREATION")
print("━" * 50)
print(f"Long-form length: {len(result.get('long_form_draft', '') or '')} chars")
print(f"Social platforms: {list(result.get('social_copies', {}).keys())[:5]}")
print(f"Visual prompts: {len(result.get('visual_prompts', []))}")

# 4. SEO
print("\n" + "━" * 50)
print("🔍 NODE 4: SEO")
print("━" * 50)
seo = result.get("seo_optimized_content", {})
print(f"Title tag: {seo.get('title_tag', 'N/A')}")
print(f"Meta description: {seo.get('meta_description', 'N/A')[:100]}...")
print(f"Readability Flesch: {seo.get('readability_score', {}).get('flesch', 'N/A')}")

# 5. QA
print("\n" + "━" * 50)
print("✅ NODE 5: QA")
print("━" * 50)
print(f"Score: {result.get('qa_score', 'N/A')}/10")
print(f"Feedback strengths: {result.get('qa_feedback', {}).get('strengths', [])[:2]}")

# 6. Distribution
print("\n" + "━" * 50)
print("📤 NODE 6: DISTRIBUTION")
print("━" * 50)
dist = result.get("distribution_plan", {})
for item in dist.get("distribution_timeline", [])[:3]:
    print(f"   Day {item.get('day')}: {item.get('channel')} — {item.get('action')}")

# 7. Analytics
print("\n" + "━" * 50)
print("📊 NODE 7: ANALYTICS")
print("━" * 50)
opt = result.get("optimization_notes", {})
print(f"What worked: {opt.get('what_worked', [])[:2]}")
print(f"Prompt improvements: {opt.get('prompt_improvements', [])[:2]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13: SAVE RESULT TO FILE
# ═══════════════════════════════════════════════════════════════

output = {
    "timestamp": datetime.now().isoformat(),
    "client_brief": result.get("client_brief"),
    "content_brief": result.get("content_brief"),
    "research_data": result.get("research_data"),
    "long_form_draft": result.get("long_form_draft"),
    "social_copies": result.get("social_copies"),
    "visual_prompts": result.get("visual_prompts"),
    "seo_optimized_content": result.get("seo_optimized_content"),
    "qa_score": result.get("qa_score"),
    "qa_feedback": result.get("qa_feedback"),
    "distribution_plan": result.get("distribution_plan"),
    "performance_metrics": result.get("performance_metrics"),
    "optimization_notes": result.get("optimization_notes")
}

# Save
with open("/mnt/agents/output/content_factory_result.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print("✅ Đã lưu kết quả: /mnt/agents/output/content_factory_result.json")
print(f"   File size: {len(json.dumps(output))} chars")